# Akan-BPE Phase 2A5 — tiny-aya-base × Akan TTS Tokenizer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/attabeezy/akan-bpe/blob/main/notebooks/2a5_tiny-aya-base_tts.ipynb)  [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/attabeezy/akan-bpe/blob/main/notebooks/2a5_tiny-aya-base_tts.ipynb)


Self-contained Colab notebook. Run all cells top-to-bottom.

**License:** CohereLabs/tiny-aya-base is CC-BY-NC (non-commercial / research use only). Accept the terms on its Hugging Face page before running.

**Steps:**
1. Clone repo and install deps
2. Download Akan datasets from HuggingFace Hub
3. Run QLoRA fine-tune experiment (`CohereLabs/tiny-aya-base` + Akan TTS tokenizer, random embedding init)
4. Inspect results — fertility reduction, **bits-per-byte (BPB, tokenizer-agnostic)**, eval loss/perplexity, generation samples
5. **Embedding-init ablation (M2)** — re-run with mean-of-subword init and compare the two arms (BPB / perplexity)

**GPU required.** Before running: Runtime → Change runtime type → T4 GPU.

> The notebook runs **two** full QLoRA fine-tunes (random + mean_subword arms), ~45–50 min each on a T4.

In [ ]:
# 1. Clone repo (skip if already inside it)
import os
from pathlib import Path

REPO = "https://github.com/attabeezy/akan-bpe.git"
REPO_NAME = "akan-bpe"

# Guard against triple-nesting on cell re-run: only clone+cd when we are not
# already sitting inside the repo root.
if Path.cwd().name != REPO_NAME:
    if not Path(REPO_NAME).is_dir():
        !git clone {REPO}
    %cd {REPO_NAME}

print(f"Working directory: {Path.cwd()}")

In [ ]:
# 2. Install dependencies
%pip install -q -e ".[dev,train]"
%pip install -q bitsandbytes sentencepiece

In [ ]:
# Hugging Face authentication — tiny-aya-base requires accepting CC-BY-NC terms.
# Accept at https://huggingface.co/CohereLabs/tiny-aya-base, then paste a token
# (or set the HF_TOKEN env var).
from huggingface_hub import login
login()

In [ ]:
# 3. Confirm GPU is available
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Go to Runtime → Change runtime type → T4 GPU, then re-run."
    )
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# 4. Download Akan datasets from HuggingFace Hub
# Produces: data/pristine_twi_{train,validation,test}.jsonl
#           data/aka_asr_{train,validation,test}.jsonl
# --tts-limit 50000 pins the pristine-Twi corpus to the same 45,000/2,500/2,500
# split used in the report (the default would pull ~188k rows and shift the split).
!python scripts/download.py --output-dir data --tts-limit 50000

In [ ]:
# 5. Train TTS tokenizer (if not already present)
from pathlib import Path

if not Path("models/tts_tokenizer.json").exists():
    print("TTS tokenizer not found — training now ...")
    !python scripts/train_bpe.py --inputs data/pristine_twi_train.jsonl --output models/tts_tokenizer.json --name tts
else:
    sz = Path("models/tts_tokenizer.json").stat().st_size / 1e6
    print(f"TTS tokenizer already present ({sz:.1f} MB), skipping.")

In [ ]:
# 5. Verify all required inputs exist
from pathlib import Path

required = {
    "TTS train data" : Path("data/pristine_twi_train.jsonl"),
    "TTS test data"  : Path("data/pristine_twi_test.jsonl"),
    "TTS tokenizer"  : Path("models/tts_tokenizer.json"),
}
missing = [name for name, p in required.items() if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing required inputs: {missing}")
print("All inputs ready.")
for name, p in required.items():
    print(f"  {name}: {p}  ({p.stat().st_size / 1e6:.1f} MB)")

## Pre-flight: tiny-aya-base architecture check

tiny-aya-base is a custom Cohere architecture. The cell below confirms the default LoRA target modules exist and reads key config. If the projection leaf names differ from the default (`q/k/v/o_proj` + `gate/up/down_proj`), add `--lora-target-modules <comma,list>` to **both** run cells below.

In [ ]:
# Pre-flight: inspect tiny-aya-base architecture before the QLoRA runs.
# The default LoRA targets are q/k/v/o_proj + gate/up/down_proj (LLaMA-style).
# If this model uses different projection leaf names, pass
# --lora-target-modules <comma,list> on BOTH run cells below.
from transformers import AutoConfig, AutoModelForCausalLM

cfg = AutoConfig.from_pretrained("CohereLabs/tiny-aya-base")
print("vocab_size        :", getattr(cfg, "vocab_size", None))
print("tie_word_embeddings:", getattr(cfg, "tie_word_embeddings", None))
print("model_type        :", getattr(cfg, "model_type", None))

# List unique linear-projection leaf names so we can confirm/adjust LoRA targets.
model = AutoModelForCausalLM.from_pretrained("CohereLabs/tiny-aya-base")
leaf_names = sorted({n.split(".")[-1] for n, _ in model.named_modules() if n.endswith("_proj")})
print("projection leaves :", leaf_names)
del model

In [ ]:
# 6. Run Phase 2A5 experiment — arm A: random embedding init (default)
# QLoRA fine-tune on tiny-aya-base with the Akan TTS tokenizer.
# Writes model adapters to models/phase2a5_tiny_aya_base_tts/
# Writes result JSON to results/phase2a5_tiny_aya_base_tts.json
# --embedding-init-mode random is the default; stated explicitly so this run pairs
# cleanly with the mean_subword ablation at the end of the notebook.
!python scripts/model_integration.py \
    --experiment-id phase2a5_tiny_aya_base_tts \
    --model-id CohereLabs/tiny-aya-base \
    --tokenizer-path models/tts_tokenizer.json \
    --train-file data/pristine_twi_train.jsonl \
    --eval-file data/pristine_twi_test.jsonl \
    --output-dir models/phase2a5_tiny_aya_base_tts \
    --results-output results/phase2a5_tiny_aya_base_tts.json \
    --device-mode colab-qlora \
    --embedding-init-mode random \
    --max-train-samples 4096 \
    --max-eval-samples 512 \
    --max-length 256 \
    --batch-size 1 \
    --grad-accum 16 \
    --epochs 1 \
    --learning-rate 2e-4 \
    --lora-r 16

In [ ]:
# 7. Load result JSON
import json
from pathlib import Path

result = json.loads(
    Path("results/phase2a5_tiny_aya_base_tts.json").read_text(encoding="utf-8")
)
print("Experiment :", result["experiment_id"])
print("Model      :", result["model_id"])
print("Device     :", result.get("device", {}))

In [ ]:
# 8. Token count comparison — fertility reduction
cmp = result["token_count_comparison"]
print(f"Base tokenizer fertility : {cmp['base_model_tokenizer']['fertility']:.3f} tokens/word")
print(f"Akan tokenizer fertility : {cmp['experiment_tokenizer']['fertility']:.3f} tokens/word")
print(f"Reduction ratio          : {cmp['token_reduction_ratio']:.1%}")

In [ ]:
# 9. Eval metrics
ev = result["eval"]
print(f"Eval loss  : {ev['eval_loss']:.4f}")
print(f"Perplexity : {ev['perplexity']:.2f}")

In [ ]:
# 9b. Bits-per-byte (BPB) — tokenizer-agnostic cross-tokenizer comparison
# Perplexity is NOT comparable across tokenizers (different vocab → different token
# counts for the same text). BPB normalizes the model's total NLL by the fixed
# UTF-8 byte count of the eval text, so the base model and the Akan-tokenizer model
# are directly comparable. Lower BPB is better; positive improvement = Akan wins.
bpb = result["eval"]["bpb"]
exp = bpb["experiment"]
base = bpb["base"]

print(f"Embedding init     : {result['embedding_init_mode']}")
print(f"Eval bytes scored  : {bpb['total_bytes']:,}")
print(f"Akan TTS model BPB : {exp['bits_per_byte']:.4f} bits/byte")
if base is not None:
    print(f"Base model BPB     : {base['bits_per_byte']:.4f} bits/byte")
    print(f"Improvement        : {bpb['improvement']:+.4f} bits/byte (base − experiment; + = Akan better)")
else:
    print("Base model BPB     : (skipped — re-run without --skip-base-bpb to populate)")

In [ ]:
# 10. Generation samples
for i, s in enumerate(result["generation_samples"], 1):
    print(f"--- Sample {i} ---")
    print(f"Prompt    : {s['prompt']}")
    print(f"Completion: {s['completion']}")
    print()

## Embedding-init ablation (M2) — random vs mean-of-subword

The run above initialized the new Akan-vocab embedding rows with the default **random**
scheme. M2 adds a modeling alternative: `--embedding-init-mode mean_subword` seeds each
Akan token's embedding from the **mean of the base model's subword embeddings** for that
token's surface string (falling back to the global embedding mean for tokens with no base
subwords). The same scheme is applied to the LM head when output embeddings are untied.

The cells below run the `mean_subword` arm as a clean A/B — identical data, tokenizer, and
hyperparameters, only the init differs — then compare the two arms on BPB and perplexity.

> **This is a second full QLoRA fine-tune (~45–50 min on a T4).** It writes a separate result
> JSON (`results/phase2a5_tiny_aya_base_tts_meansub.json`) and adapter dir, so the random-arm
> artifacts above are left untouched.

In [ ]:
# 11. Ablation arm B: mean-of-subword embedding init (second full QLoRA run)
!python scripts/model_integration.py \
    --experiment-id phase2a5_tiny_aya_base_tts_meansub \
    --model-id CohereLabs/tiny-aya-base \
    --tokenizer-path models/tts_tokenizer.json \
    --train-file data/pristine_twi_train.jsonl \
    --eval-file data/pristine_twi_test.jsonl \
    --output-dir models/phase2a5_tiny_aya_base_tts_meansub \
    --results-output results/phase2a5_tiny_aya_base_tts_meansub.json \
    --device-mode colab-qlora \
    --embedding-init-mode mean_subword \
    --max-train-samples 4096 \
    --max-eval-samples 512 \
    --max-length 256 \
    --batch-size 1 \
    --grad-accum 16 \
    --epochs 1 \
    --learning-rate 2e-4 \
    --lora-r 16

In [ ]:
# 12. Compare embedding-init arms: random (arm A) vs mean_subword (arm B)
import json
from pathlib import Path


def load_eval(path):
    r = json.loads(Path(path).read_text(encoding="utf-8"))
    b = r["eval"]["bpb"]
    return {
        "init": r["embedding_init_mode"],
        "rows_initialized": (r.get("embedding_init") or {}).get("rows_initialized"),
        "eval_loss": r["eval"]["eval_loss"],
        "perplexity": r["eval"]["perplexity"],
        "exp_bpb": b["experiment"]["bits_per_byte"],
        "base_bpb": (b["base"] or {}).get("bits_per_byte"),
        "improvement": b.get("improvement"),
    }


arms = {
    "random": load_eval("results/phase2a5_tiny_aya_base_tts.json"),
    "mean_subword": load_eval("results/phase2a5_tiny_aya_base_tts_meansub.json"),
}

header = f"{'metric':<24}{'random':>14}{'mean_subword':>16}"
print(header)
print("-" * len(header))


def show(label, key, fmt="{:.4f}"):
    cells = []
    for arm in ("random", "mean_subword"):
        v = arms[arm][key]
        cells.append(fmt.format(v) if v is not None else "—")
    print(f"{label:<24}{cells[0]:>14}{cells[1]:>16}")


show("eval_loss", "eval_loss")
show("perplexity", "perplexity", "{:.2f}")
show("experiment BPB", "exp_bpb")
show("base BPB", "base_bpb")
show("improvement (base−exp)", "improvement")

best = min(arms, key=lambda a: arms[a]["exp_bpb"])
print(f"\nLower fine-tuned BPB: {best!r} arm.")